# DermAI: Phase 1 — EfficientNet-B4 Training on HAM10000

This notebook downloads the 10,015 skin lesion images from the [HAM10000 dataset](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000), processes them, and fine-tunes an `EfficientNet-B4` architecture to map images to real clinical probability scores for 7 lesion classes.

**Pre-requisites:**
1. Go to `Runtime > Change runtime type` and select **T4 GPU** (or better).
2. Have a Kaggle account ready (free signup at kaggle.com).

In [ ]:
# 1. Install dependencies
!pip install timm torch torchvision pandas numpy scikit-learn pillow matplotlib kaggle -q
print('✅ Dependencies installed!')

### 🔑 Setup Kaggle API
1. Go to [kaggle.com](https://kaggle.com) → Profile icon → **Settings**
2. Scroll to **API** section → Click **"Create New API Token"**
3. Copy the token that looks like: `KGAT_895c03e1cfbd...`
4. Paste it below ⬇️

In [ ]:
import os

# ╔═══════════════════════════════════════════════════════════╗
# ║  PASTE YOUR KAGGLE API TOKEN BELOW                       ║
# ╚═══════════════════════════════════════════════════════════╝
KAGGLE_TOKEN = ""  # ← paste your KGAT_... token here

# Set the environment variable (this is how the new Kaggle CLI authenticates)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

if KAGGLE_TOKEN.startswith('KGAT_'):
    print(f'✅ Kaggle API token set! (starts with {KAGGLE_TOKEN[:10]}...)')
else:
    print('⚠️  WARNING: Paste your KGAT_... token above and re-run this cell!')

In [ ]:
# 2. Download the HAM10000 Dataset (~3GB, takes 2-3 min)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d dataset/
print('✅ Dataset extracted to /dataset/')

In [ ]:
# 3. Import Libraries & Set Config
import torch, timm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from glob import glob

class Config:
    SEED = 42
    IMG_SIZE = 380  # Ideal for EfficientNet-B4
    BATCH_SIZE = 16 # Adjust if you run out of GPU memory
    EPOCHS = 10
    LR = 3e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_NAME = 'efficientnet_b4'

torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)
print(f'✅ Using device: {Config.DEVICE}')
if str(Config.DEVICE) != 'cuda':
    print('⚠️  WARNING: GPU not detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# 4. Prepare Metadata & Images
df = pd.read_csv('dataset/HAM10000_metadata.csv')
image_paths = {os.path.basename(x).split('.')[0]: x for x in glob('dataset/*/*.jpg')}
df['path'] = df['image_id'].map(image_paths)

# Drop rows where path is NaN
df.dropna(subset=['path'], inplace=True)

# Encode classes
classes = sorted(df['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df['label'] = df['dx'].map(class_to_idx)

print(f'✅ Found {len(df)} images across {len(classes)} classes:')
print(df['dx'].value_counts())

In [ ]:
# 5. Handling Imbalance & Splitting
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=Config.SEED)

# Calculate weights for WeightedRandomSampler
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = np.array([class_weights[y] for y in train_df['label']])
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
print(f'✅ Train: {len(train_df)} images | Val: {len(val_df)} images')

In [ ]:
# 6. Dataset Class & Augmentation
class DermDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.transforms = transforms
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        label = row['label']
        if self.transforms:
            img = self.transforms(img)
        return img, torch.tensor(label, dtype=torch.long)

# Augmentations
train_transforms = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(DermDataset(train_df, train_transforms), batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(DermDataset(val_df, val_transforms), batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
print(f'✅ DataLoaders ready — Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# 7. Define the Model Architecture
# THIS MUST MATCH YOUR server.py BACKEND EXACTLY
class DermAIClassifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0, global_pool='')
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc      = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.fc(self.dropout(self.pool(f).flatten(1)))

model = DermAIClassifier(num_classes=len(classes)).to(Config.DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=Config.LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
print('✅ Model loaded on', Config.DEVICE)

In [ ]:
# 8. 🔥 TRAINING (takes ~45-60 min — sit back and relax!)
best_acc = 0.0

for epoch in range(Config.EPOCHS):
    model.train()
    train_loss = 0.0
    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(Config.DEVICE), labels.to(Config.DEVICE)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * imgs.size(0)

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(Config.DEVICE), labels.to(Config.DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    train_loss = train_loss / len(train_df)
    val_loss = val_loss / len(val_df)
    val_acc = correct / total
    scheduler.step(val_acc)
    
    print(f'Epoch {epoch+1}/{Config.EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    
    if val_acc > best_acc:
        best_acc = val_acc
        # Save the model state including metadata required by server.py
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'classes': classes,
            'class_names': {
                'nv':    'Melanocytic Nevi',
                'mel':   'Melanoma',
                'bkl':   'Benign Keratosis',
                'bcc':   'Basal Cell Carcinoma',
                'akiec': 'Actinic Keratosis',
                'vasc':  'Vascular Lesion',
                'df':    'Dermatofibroma'
            },
            'malignant_classes': ['mel','bcc','akiec'],
            'idx_to_class': {i: c for i, c in enumerate(classes)}
        }
        torch.save(checkpoint, 'dermai_model_full.pth')
        print('🔥 Best model saved.')

print(f'\n✅ Training complete! Best validation accuracy: {best_acc:.4f}')

In [ ]:
# 9. Download your trained model to your local machine
from google.colab import files
files.download('dermai_model_full.pth')

print('✅ Once downloaded, place this file inside your project:')
print('   dermai/backend/models/dermai_model_full.pth')
print('   Then restart the backend — it will load the REAL model! 🎉')